# 07 — Experiment Summary and Paired Statistical Analysis

This notebook is the final offline reporting stage. It reads only persisted paired GLueCoS fine-tuning results, verifies complete seed coverage and aligned prediction references, computes the configured paired seed-level and bootstrap tests, and saves both machine-readable JSON and a human-readable Markdown experiment summary. It does not train, modify checkpoints, or fetch data.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def find_project_root() -> Path:
    configured_root = os.environ.get('PROJECT_ROOT')
    starting_points = [Path(configured_root)] if configured_root else [Path.cwd()]
    for starting_point in starting_points:
        resolved_start = starting_point.expanduser().resolve()
        for candidate in (resolved_start, *resolved_start.parents):
            if (candidate / 'configs' / 'config.yaml').is_file() and (candidate / 'INSTRUCTIONS.md').is_file():
                return candidate
    raise FileNotFoundError('Could not locate PROJECT_ROOT. Launch Jupyter from the repository root or set PROJECT_ROOT.')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'PROJECT_ROOT: {PROJECT_ROOT}')


## 1. Load reproducible configuration and result locations

The configured seed list, confidence level, bootstrap count, and significance threshold are reported directly from YAML. The analysis uses `probabilistic minus BPE`, so a positive difference favors the proposed system.

In [ ]:
import json

import yaml

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'config.yaml'
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
required_sections = {'paths', 'experiment', 'evaluation', 'data_download'}
missing_sections = required_sections.difference(config)
if missing_sections:
    raise KeyError(f'Missing configuration sections: {sorted(missing_sections)}')

EXPERIMENT_DIRECTORY = PROJECT_ROOT / config['paths']['experiments'] / config['experiment']['id']
DOWNSTREAM_RESULTS_PATH = EXPERIMENT_DIRECTORY / 'downstream_metrics.json'
STATISTICS_DIRECTORY = PROJECT_ROOT / config['paths']['results'] / 'statistics'
print(json.dumps({
    'experiment_id': config['experiment']['id'],
    'seeds': config['evaluation']['seeds'],
    'confidence_level': config['evaluation']['confidence_level'],
    'paired_bootstrap_samples': config['evaluation']['paired_bootstrap_samples'],
    'significance_level': config['evaluation']['significance_level'],
    'downstream_results': str(DOWNSTREAM_RESULTS_PATH),
}, indent=2))


## 2. Preflight complete paired inputs

All configured GLueCoS repositories and all configured seeds must be present before a result can be reported. The source analysis function independently checks that BPE and probabilistic labels are identical before running a paired bootstrap.

In [ ]:
if not DOWNSTREAM_RESULTS_PATH.is_file():
    raise FileNotFoundError(f'Run 04_gluecos_evaluation.ipynb first: {DOWNSTREAM_RESULTS_PATH}')

downstream_results = json.loads(DOWNSTREAM_RESULTS_PATH.read_text(encoding='utf-8'))
expected_repositories = set(config['data_download']['gluecos'])
missing_repositories = expected_repositories.difference(downstream_results)
if missing_repositories:
    raise ValueError(f'Missing GLueCoS results for: {sorted(missing_repositories)}')

expected_seeds = {str(int(seed)) for seed in config['evaluation']['seeds']}
for repository in sorted(expected_repositories):
    observed_seeds = set(downstream_results[repository]['seeds'])
    if observed_seeds != expected_seeds:
        raise ValueError(f'{repository}: expected seeds {sorted(expected_seeds)}, found {sorted(observed_seeds)}')
print(f'Preflight passed: {len(expected_repositories)} tasks × {len(expected_seeds)} paired seeds.')


## 3. Generate all paired statistical outputs

For every shared metric, the report contains per-arm mean, sample standard deviation, t-based confidence interval, and paired seed-level t-test. It also computes a paired bootstrap accuracy difference from aligned saved predictions for the first configured seed. The result is saved atomically under both `outputs/results/statistics/` and the self-contained experiment directory.

In [ ]:
from src.evaluation.analysis import analyze_finetuning_results, write_experiment_summary

statistical_report = analyze_finetuning_results(PROJECT_ROOT, config, downstream_results)
summary_paths = write_experiment_summary(PROJECT_ROOT, config, statistical_report)
print(json.dumps(summary_paths, indent=2))


## 4. Review the saved report

The table below is rendered from the generated Markdown summary, not recomputed in the notebook. Read confidence intervals and p-values together; a mean difference alone is not a significance claim.

In [ ]:
from IPython.display import Markdown, display

summary_markdown_path = Path(summary_paths['results_summary'])
display(Markdown(summary_markdown_path.read_text(encoding='utf-8')))

statistics_json_path = STATISTICS_DIRECTORY / 'paired_gluecos_analysis.json'
print(f'Paired JSON report: {statistics_json_path}')
print(f"Experiment-local JSON report: {EXPERIMENT_DIRECTORY / 'statistical_analysis.json'}")
print(f"Experiment-local Markdown summary: {summary_paths['experiment_summary']}")


## Reporting checklist

Use the saved JSON for machine-readable results and the Markdown summary for a concise report. Any conclusion should identify the task, metric, seed summary, confidence interval, paired seed-test p-value, and paired-bootstrap result; results remain exploratory unless the configured `p < 0.05` criterion is met.